# 1D-CNN — Optuna Tuning

## Imports

In [ ]:
import optuna
import numpy as np
import torch
import torch.nn as nn
import os
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

work_dir = os.getcwd(); print(work_dir)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Data Loader

In [ ]:
input_folder = os.path.join(work_dir, "data_cnn_normalized")

X_train = np.load(os.path.join(input_folder, "X_train.npy"))
y_train = np.load(os.path.join(input_folder, "y_train.npy"))
X_val   = np.load(os.path.join(input_folder, "X_val.npy"))
y_val   = np.load(os.path.join(input_folder, "y_val.npy"))
X_test  = np.load(os.path.join(input_folder, "X_test.npy"))
y_test  = np.load(os.path.join(input_folder, "y_test.npy"))

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"y_train: [{y_train.min():.2f}, {y_train.max():.2f}]")

def create_loader(X, y, batch_size=128, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)

train_loader = create_loader(X_train, y_train)
val_loader   = create_loader(X_val,   y_val,  shuffle=False)
test_loader  = create_loader(X_test,  y_test, shuffle=False)


## Model

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, input_channels, base_channels=64, dropout=0.2):
        super().__init__()
        bc = base_channels
        self.net = nn.Sequential(
            nn.Conv1d(input_channels, bc,    7, padding=3), nn.BatchNorm1d(bc),    nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(bc,             bc*2,  5, padding=2), nn.BatchNorm1d(bc*2),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(bc*2,           bc*4,  3, padding=1), nn.BatchNorm1d(bc*4),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(bc*4,           bc*2,  3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(bc*2, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)   # (B, C, T)
        x = self.net(x).squeeze(-1)
        return self.fc(self.dropout(x)).squeeze(-1)

CNN1D_MODEL_CLASS = CNN1D  # alias for CV block

model = CNN1D(input_channels=X_train.shape[2]).to(DEVICE)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"1D-CNN params: {total:,}")


## Objective

In [ ]:
def objective(trial):
    base_channels = trial.suggest_int("base_channels", 16, 96)
    dropout       = trial.suggest_float("dropout", 0.0, 0.4)
    lr            = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [64, 128, 256])
    weight_decay  = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    huber_delta   = trial.suggest_float("huber_delta", 1.0, 5.0)

    _train = create_loader(X_train, y_train, batch_size)
    _val   = create_loader(X_val,   y_val,   batch_size, shuffle=False)

    m    = CNN1D(X_train.shape[2], base_channels=base_channels, dropout=dropout).to(DEVICE)
    opt  = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.HuberLoss(delta=huber_delta)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)

    best_val, counter = float("inf"), 0
    for epoch in range(50):
        m.train()
        for xb, yb in _train:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        m.eval(); vl = []
        with torch.no_grad():
            for xb, yb in _val:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vl.append(crit(m(xb), yb).item())
        val_loss = np.mean(vl)
        sch.step(val_loss)
        trial.report(val_loss, epoch)
        if trial.should_prune(): raise optuna.exceptions.TrialPruned()
        if val_loss < best_val: best_val, counter = val_loss, 0
        else: counter += 1
        if counter >= 7: break
    return best_val


## Study

In [ ]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study.optimize(objective, n_trials=50)

## Results

In [ ]:
print('Best Loss  :', study.best_value)
print('Best Params:', study.best_params)
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.tight_layout(); plt.show()
optuna.visualization.matplotlib.plot_param_importances(study)
plt.tight_layout(); plt.show()